In [1]:
import os

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str  # <--- Verify this name
    local_data_file: Path
    unzip_dir: Path
    


In [3]:
import os
import sys

# Print current directory to be 100% sure
print(f"Current Directory: {os.getcwd()}")

# Check if 'src' actually exists here
if os.path.exists("src"):
    print("Found 'src' folder!")
    # Check if 'textsummarizer' exists inside 'src'
    if os.path.exists("src/textsummarizer"):
        print("Found 'textsummarizer' inside 'src'!")
    else:
        print("Error: 'textsummarizer' folder not found inside 'src'")
else:
    print("Error: 'src' folder not found in current directory")

# Re-append with absolute path to be safe
sys.path.append(os.path.abspath("src"))

Current Directory: c:\Users\chauh\OneDrive\Desktop\Text-Summarization\research
Error: 'src' folder not found in current directory


In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\chauh\\OneDrive\\Desktop\\Text-Summarization'

In [6]:
# import sys
# import os

# # 1. Ensure you are in the project root
# # (Replace this with your project path if %pwd doesn't show the project folder)
# os.chdir(r'C:\Users\chauh\OneDrive\Desktop\Text-Summarization')

# # 2. Tell Python to look into the 'src' folder for modules
# sys.path.append("src")

# 3. Now try your imports
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE__PATH):
        
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)


        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config=self.config.data_ingestion

        create_directories([config.root_dir])

        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url, # <--- Make sure this is 'source_url'
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config
    
        

In [8]:
import os
import urllib.request as request
import zipfile
from textsummarizer.logging import logger
from textsummarizer.utils.common import get_size




In [9]:
from box import ConfigBox
import yaml
from typeguard import typechecked

@typechecked
def read_yaml(path_to_yaml: Path) -> ConfigBox:
    try:
        with open(path_to_yaml) as yaml_file:
            content = yaml.safe_load(yaml_file)
            return ConfigBox(content) # Ensure you are wrapping it here!
    except Exception as e:
        raise e

In [ ]:
import os
import zipfile
import requests
from pathlib import Path
from textsummarizer.logging import logger

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
   
   
    def download_file(self):
        if os.path.exists(self.config.local_data_file):
            file_size = os.path.getsize(self.config.local_data_file)
            # If the file is the "bad" 23MB size or too small, delete it
            if file_size == 23627009 or file_size < 1000:
                logger.info(f"Detected corrupted file ({file_size} bytes). Deleting...")
                os.remove(self.config.local_data_file)

        if not os.path.exists(self.config.local_data_file):
            logger.info(f"Downloading fresh data from: {self.config.source_url}")
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(self.config.source_url, headers=headers, stream=True)
            
            if response.status_code == 200:
                with open(self.config.local_data_file, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=1024):
                        f.write(chunk)
                logger.info(f"Download complete! Size: {os.path.getsize(self.config.local_data_file)} bytes")
            else:
                logger.error(f"Download failed! Status: {response.status_code}")




    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        
        # DEBUG LINES
        print(f"Targeting file: {os.path.abspath(self.config.local_data_file)}")
        print(f"Actual File Size: {os.path.getsize(self.config.local_data_file)} bytes")
        
        with open(self.config.local_data_file, 'rb') as f:
            if f.read(2) != b'PK':
                raise ValueError("Not a valid ZIP file!")
        
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Extracted to {unzip_path}")

In [11]:
# try:
#     config=ConfigurationManager()
#     data_ingestion_config=config.get_data_ingestion_config()
#     data_ingestion=DataIngestion(config=data_ingestion_config)
#     data_ingestion.download_file()
#     data_ingestion.extract_zip_file()

# except Exception as e:
#     raise e


try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    
    # Check if file exists before downloading
    if not os.path.exists(data_ingestion_config.local_data_file):
        data_ingestion.download_file()
    else:
        print("File already exists, skipping download and moving to extraction.")
        
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-04-28 19:43:45,153: INFO:  common: created directory at: artifacts ]
[2026-04-28 19:43:45,156: INFO:  common: created directory at: artifacts/data_ingestion ]


File already exists, skipping download and moving to extraction.
Targeting file: c:\Users\chauh\OneDrive\Desktop\Text-Summarization\artifacts\data_ingestion\data.zip
Actual File Size: 7903594 bytes
[2026-04-28 19:43:45,372: INFO:  993747597: Extracted to artifacts/data_ingestion ]
